### Complete Order (Delay)
For delay-based objectives, two separation-identical aircraft $i$ and $j$ induce a complete order when $r_i \leq r_j$ and $b_i \leq b_j$. For any partial sequence, placing $i$ before $j$ yields a schedule whose delay is no worse than the schedule obtained by swapping them, for every aircraft other than $i$ and $j$. The cell below verifies both the per-aircraft and total-delay variants of this dominance relation.


In [ ]:
from cvc5 import Kind
from core.context import *
from core.checks import *

# Construct the two sequences that differ only in the relative order of i and j.
S1, S2, ctx = get_sequences(integer_arithmetic=True)
solver = ctx.solver

# State the premises: order the release and baseline times, and require i and j to be separation-equivalent.
premises = [
    solver.mkTerm(Kind.LEQ, ctx.r["i"], ctx.r["j"]),  solver.mkTerm(Kind.LEQ, ctx.b["i"], ctx.b["j"]),
    *ctx.separation_equivalence("i", "j"),
]

# Verify that every aircraft other than i and j incurs no larger delay in S1.
claim = solver.mkTerm(Kind.AND, *[
    solver.mkTerm(Kind.LEQ, S1.delay[x], S2.delay[x])
    for x in ctx.aircraft if x not in ("i", "j")
])

res = verify_pruning_rule(ctx, premises, claim)
print(f"\nComplete Order Delay (per-aircraft): {res}\n")

# Verify that total delay in S1 is no larger than in S2.
claim = solver.mkTerm(
    Kind.LEQ,
    solver.mkTerm(Kind.ADD, *[S1.delay[x] for x in ctx.aircraft]),
    solver.mkTerm(Kind.ADD, *[S2.delay[x] for x in ctx.aircraft]),
)

res = verify_pruning_rule(ctx, premises, claim)
print(f"\nComplete Order Delay (total):        {res}\n")


### Complete Order (Makespan)
For makespan minimisation, two separation-identical aircraft $i$ and $j$ induce a complete order when $r_i \leq r_j$. In any partial sequence, placing $i$ before $j$ produces a makespan no worse than the one obtained by swapping their positions. The cell below verifies this dominance argument directly.

In [ ]:
from cvc5 import Kind
from core.context import *
from core.checks import *

# Construct the two sequences that differ only in the relative order of i and j.
S1, S2, ctx = get_sequences(integer_arithmetic=True)
solver = ctx.solver

# State the premises: order the release times and require i and j to be separation-equivalent.
premises = [
    solver.mkTerm(Kind.LEQ, ctx.r["i"], ctx.r["j"]),
    *ctx.separation_equivalence("i", "j"),
]

# Verify that the makespan of S1 is no larger than that of S2.
claim = solver.mkTerm(Kind.LEQ, S1.makespan, S2.makespan)

res = verify_pruning_rule(ctx, premises, claim)
print(f"\nComplete Order Makespan: {res}\n")


### Complete Order (Time Window Feasibility)
For time-window feasibility, two separation-identical aircraft $i$ and $j$ induce a complete order when $r_i \leq r_j$ and $lt_i \leq lt_j$. If the sequence with $j$ before $i$ is feasible, then swapping them so that $i$ precedes $j$ does not create any new time-window violations. The cell below verifies this preservation result.


In [ ]:
from cvc5 import Kind
from core.context import *
from core.checks import *

# Construct the two sequences that differ only in the relative order of i and j.
S1, S2, ctx = get_sequences(integer_arithmetic=True)
solver = ctx.solver

# State the premises: order the release and latest times, require separation equivalence, and assume S2 is feasible.
premises = [
    solver.mkTerm(Kind.LEQ, ctx.r["i"], ctx.r["j"]),  solver.mkTerm(Kind.LEQ, ctx.lt["i"], ctx.lt["j"]),
    *ctx.separation_equivalence("i", "j"),  *S2.time_window_feasible,
]

# Verify that S1 does not introduce any time-window violation absent from S2.
claim = solver.mkTerm(Kind.AND, *[
    solver.mkTerm(Kind.OR, solver.mkTerm(Kind.NOT, S2.window_violation[x]), S1.window_violation[x])
    for x in ctx.aircraft
])

res = verify_pruning_rule(ctx, premises, claim)
print(f"\nComplete Order Time Windows: {res}\n")


### Counterexample (CTOT)
Since a complete-order cannot be infered with regards to CTOT penalty, attempts to find this would result in a counterexample. The cell below shows this.

In [ ]:
from cvc5 import Kind
from core.context import *
from core.checks import *

# Construct the two sequences that differ only in the relative order of i and j.
S1, S2, ctx = get_sequences(integer_arithmetic=True)
solver = ctx.solver

# State the premises: order the release and baseline times, and require i and j to be separation-equivalent.
premises = [
    solver.mkTerm(Kind.LEQ, ctx.r["i"], ctx.r["j"]),  solver.mkTerm(Kind.LEQ, ctx.b["i"], ctx.b["j"]),
    *ctx.separation_equivalence("i", "j"),
]

# Verify that total CTOT penalty in S1 is no larger than in S2.
claim = solver.mkTerm(
    Kind.LEQ,
    solver.mkTerm(Kind.ADD, *[S1.ctot[x] for x in ctx.aircraft]),
    solver.mkTerm(Kind.ADD, *[S2.ctot[x] for x in ctx.aircraft]),
)

res = verify_pruning_rule(ctx, premises, claim)
print(f"\nComplete Order CTOT (total):        {res}\n")
